# PHQ-8 Clinical Relevance Analysis

R3: Clinical relevance of discordance.

### Required outputs
1. Participant-level mean SADI from the correctly paired held-out E-DAIC test set.
2. PHQ-8 total and prespecified PHQ-8 ≥ 10 grouping.
3. Spearman's ρ between participant-level mean SADI and PHQ-8 total.
4. 95% CI for Spearman's ρ using participant-level bootstrap resampling.
5. Mann–Whitney U comparison of SADI between PHQ-8 < 10 and PHQ-8 ≥ 10.
6. Cliff's δ effect size.
7. Figure 5: (a) SADI vs PHQ-8 with LOESS and PHQ-8=10 cutoff; (b) violin plots by PHQ-8 group.
8. CSV, JSON, PNG, SVG, and manuscript-ready text outputs.

**Important:** Deliberately mismatched perturbation samples are not used here. This analysis asks whether naturally occurring SADI in correctly paired E-DAIC test segments relates to symptom severity.

The segment-level SADI are averaged per participant and related to PHQ-8 total using **Spearman's ρ**, with a 95% CI, p-value, and n. A **Mann–Whitney** comparison of PHQ-8 ≥ 10 versus <10, reporting group medians and **Cliff's δ**, plus a two-panel Figure. A null association must be reported honestly rather than softened.

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu
from statsmodels.nonparametric.smoothers_lowess import lowess

CURRENT_DIR = Path.cwd().resolve()

# If the notebook is running from e_daic/notebooks,
# the project root is one level above.
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    # Otherwise search upwards for the project structure
    PROJECT_ROOT = None

    for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "models").is_dir()
            and (candidate / "src").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            PROJECT_ROOT = candidate
            break

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Could not locate the e_daic project root."
        )

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

TEST_PATH = PROJECT_ROOT / "data/processed/quantum/test.pkl"
MANIFEST_PATH = PROJECT_ROOT / "data/processed/official_split/participant_manifest.csv"
V6_SEED = 2024
CHECKPOINT_PATH = PROJECT_ROOT / f"models/v6_seed_{V6_SEED}_best.pt"

OUTPUT_DIR = NOTEBOOKS_DIR / "phq_analyse"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PHQ_CUTOFF = 10
BOOTSTRAPS = 10_000
BOOTSTRAP_SEED = 2025
BATCH_SIZE = 8
RANK = 16

print("Project root:", PROJECT_ROOT.resolve())
print("Output folder:", OUTPUT_DIR.resolve())
print("Test data:", TEST_PATH.resolve())
print("Manifest:", MANIFEST_PATH.resolve())
print("Checkpoint:", CHECKPOINT_PATH.resolve())

## Load the official E-DAIC test participant metadata

The official participant split is authoritative. The PHQ analysis uses the **56 held-out test participants**. For the clinical group comparison, the requested rule is applied directly to the continuous PHQ-8 total: **PHQ-8 ≥ 10 = depressed; PHQ-8 < 10 = non-depressed**. This is separate from any legacy binary label field.

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

manifest = pd.read_csv(MANIFEST_PATH, dtype={"participant_id": str})
required = {"participant_id", "split", "phq8_total"}
missing = required - set(manifest.columns)
if missing:
    raise ValueError(f"Manifest missing required columns: {sorted(missing)}")

test_manifest = manifest.loc[
    manifest["split"].astype(str).str.lower().eq("test")
].copy()
test_manifest["participant_id"] = test_manifest["participant_id"].astype(str)
test_manifest["phq8_total"] = pd.to_numeric(test_manifest["phq8_total"], errors="coerce")
test_manifest["phq8_group"] = np.where(
    test_manifest["phq8_total"] >= PHQ_CUTOFF,
    "PHQ-8 ≥ 10",
    "PHQ-8 < 10",
)

assert len(test_manifest) == 56, f"Expected 56 test participants, found {len(test_manifest)}"
assert test_manifest["participant_id"].is_unique
assert test_manifest["phq8_total"].notna().all()

print("Official participants:", len(manifest))
print("Test participants:", len(test_manifest))
display(test_manifest.head())
print("\nPHQ-8 groups:")
display(test_manifest["phq8_group"].value_counts().rename_axis("group").reset_index(name="n"))

## Load the proposed model and evaluate correctly paired test segments

The implementation returns `sadi` from the post-interaction reduced states. Segment-level SADI is then averaged within participant, exactly as requested by R3.

In [6]:
import torch
from torch.utils.data import DataLoader
from src.proposed.dataset import QuantumSequenceDataset, collate_fn
from src.proposed.v6.model_v6 import ProposedModelV6

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

if not TEST_PATH.exists():
    raise FileNotFoundError(TEST_PATH)
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)

test_dataset = QuantumSequenceDataset(TEST_PATH)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model = ProposedModelV6(rank=RANK).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Test records:", len(test_dataset))
print("Checkpoint seed:", checkpoint.get("seed"))
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Development threshold:", checkpoint.get("dev_threshold"))

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Test records: 5081
Checkpoint seed: 2024
Checkpoint epoch: 1
Development threshold: 0.37499999999999994


In [ ]:
segment_rows = []
start = time.time()

with torch.no_grad():
    for batch in test_loader:
        audio = batch["audio"].to(DEVICE)
        text = batch["text"].to(DEVICE)
        audio_mask = batch["audio_mask"].to(DEVICE)
        text_mask = batch["text_mask"].to(DEVICE)

        output = model(audio, text, audio_mask, text_mask)

        sadi = output["sadi"].detach().cpu().numpy()
        fidelity = output["fidelity"].detach().cpu().numpy()
        trace_distance = output["trace_distance"].detach().cpu().numpy()
        qmi = output["qmi"].detach().cpu().numpy()

        for i in range(len(sadi)):
            segment_rows.append({
                "participant_id": str(batch["participant_ids"][i]),
                "utterance_id": str(batch["utterance_ids"][i]),
                "sadi": float(sadi[i]),
                "fidelity": float(fidelity[i]),
                "trace_distance": float(trace_distance[i]),
                "qmi": float(qmi[i]),
            })

segment_sadi_df = pd.DataFrame(segment_rows)
print(f"Segments evaluated: {len(segment_sadi_df):,}")
print("Participants:", segment_sadi_df["participant_id"].nunique())
print(f"Inference time: {time.time()-start:.2f} s")

assert segment_sadi_df["participant_id"].nunique() == 56
assert np.isfinite(segment_sadi_df[["sadi","fidelity","trace_distance","qmi"]].to_numpy()).all()
display(segment_sadi_df.head())

## Participant-level SADI and PHQ-8 analysis dataset

The primary analysis unit is the participant. All segment-level SADI values are averaged within participant before any PHQ-8 statistical test.

In [ ]:
participant_sadi = (
    segment_sadi_df.groupby("participant_id", as_index=False)
    .agg(
        mean_sadi=("sadi", "mean"),
        median_sadi=("sadi", "median"),
        n_segments=("sadi", "size"),
        mean_fidelity=("fidelity", "mean"),
        mean_trace_distance=("trace_distance", "mean"),
        mean_qmi=("qmi", "mean"),
    )
)

participant_sadi["participant_id"] = participant_sadi["participant_id"].astype(str)

phq_df = participant_sadi.merge(
    test_manifest[["participant_id", "phq8_total", "phq8_group"]],
    on="participant_id",
    how="inner",
    validate="one_to_one",
)

assert len(phq_df) == 56
assert phq_df["participant_id"].is_unique
assert phq_df["phq8_total"].notna().all()
assert phq_df["mean_sadi"].notna().all()

print("Participant-level rows:", len(phq_df))
display(phq_df.head())

a = TABLE_DIR / "phq_participant_level.csv"
b = TABLE_DIR / "phq_segment_level_sadi.csv"
phq_df.to_csv(a, index=False)
segment_sadi_df.to_csv(b, index=False)
print("Saved:", a.resolve())
print("Saved:", b.resolve())

## Spearman association: participant-level SADI vs PHQ-8 total

Spearman's ρ, p, n, and a 95% CI. The CI is obtained by **participant-level bootstrap resampling (10,000 resamples)** so segments from the same participant are never treated as independent.

In [ ]:
rho_result = spearmanr(phq_df["phq8_total"], phq_df["mean_sadi"])
rho = float(rho_result.statistic)
p_value = float(rho_result.pvalue)

print(f"Spearman rho: {rho:.6f}")
print(f"p-value:      {p_value:.6g}")
print(f"n:            {len(phq_df)}")

In [ ]:
def bootstrap_spearman_ci(df, x_col, y_col, n_bootstrap=10_000, seed=2025, confidence=0.95):
    rng = np.random.default_rng(seed)
    x = df[x_col].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)
    n = len(df)
    boot = np.empty(n_bootstrap, dtype=float)
    valid = 0
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        value = float(spearmanr(x[idx], y[idx]).statistic)
        if np.isfinite(value):
            boot[valid] = value
            valid += 1
    boot = boot[:valid]
    alpha = 1 - confidence
    return float(np.quantile(boot, alpha/2)), float(np.quantile(boot, 1-alpha/2)), valid

rho_lo, rho_hi, n_valid_boot = bootstrap_spearman_ci(
    phq_df, "phq8_total", "mean_sadi", BOOTSTRAPS, BOOTSTRAP_SEED
)

## PHQ-8 ≥ 10 group comparison

Prespecified groups:

- **PHQ-8 < 10:** non-depressed
- **PHQ-8 ≥ 10:** depressed

Required statistics: median SADI in each group, Mann–Whitney U and p-value, and Cliff's δ.

In [11]:
non_depressed = phq_df.loc[phq_df["phq8_group"].eq("PHQ-8 < 10"), "mean_sadi"].to_numpy(float)
depressed = phq_df.loc[phq_df["phq8_group"].eq("PHQ-8 ≥ 10"), "mean_sadi"].to_numpy(float)

assert len(non_depressed) > 0 and len(depressed) > 0

mw = mannwhitneyu(depressed, non_depressed, alternative="two-sided")
mw_u = float(mw.statistic)
mw_p = float(mw.pvalue)


def cliffs_delta(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    d = x[:, None] - y[None, :]
    return float((np.sum(d > 0) - np.sum(d < 0)) / d.size)

cliffs_d = cliffs_delta(depressed, non_depressed)

dep_median = float(np.median(depressed))
non_dep_median = float(np.median(non_depressed))

print("Non-depressed n:", len(non_depressed))
print("Depressed n:", len(depressed))
print(f"Median SADI, PHQ-8 < 10: {non_dep_median:.6f}")
print(f"Median SADI, PHQ-8 ≥ 10: {dep_median:.6f}")
print(f"Mann–Whitney U: {mw_u:.3f}")
print(f"Mann–Whitney p: {mw_p:.6g}")
print(f"Cliff's delta: {cliffs_d:.6f}")

group_results = pd.DataFrame([{
    "comparison": "PHQ-8 ≥ 10 vs PHQ-8 < 10",
    "non_depressed_n": len(non_depressed),
    "depressed_n": len(depressed),
    "non_depressed_median_sadi": non_dep_median,
    "depressed_median_sadi": dep_median,
    "mann_whitney_U": mw_u,
    "mann_whitney_p": mw_p,
    "cliffs_delta": cliffs_d,
}])
display(group_results)

Non-depressed n: 35
Depressed n: 21
Median SADI, PHQ-8 < 10: 0.990658
Median SADI, PHQ-8 ≥ 10: 0.991725
Mann–Whitney U: 453.000
Mann–Whitney p: 0.150275
Cliff's delta: 0.232653


,comparison,non_depressed_n,depressed_n,non_depressed_median_sadi,depressed_median_sadi,mann_whitney_U,mann_whitney_p,cliffs_delta
0,PHQ-8 ≥ 10 vs PHQ-8 < 10,35,21,0.990658,0.991725,453.0,0.150275,0.232653


## Figure: clinical relevance of SADI

Panel (a): participant-level SADI against PHQ-8 total, with LOESS and the PHQ-8=10 cutoff.

Panel (b): violin plots of participant-level SADI for PHQ-8 < 10 and PHQ-8 ≥ 10.

The figure is exported at 600 dpi and SVG.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))

# Panel A
ax = axes[0]
x = phq_df["phq8_total"].to_numpy(float)
y = phq_df["mean_sadi"].to_numpy(float)
ax.scatter(x, y, s=34, alpha=0.8)
loess_fit = lowess(y, x, frac=0.65, it=0, return_sorted=True)
ax.plot(loess_fit[:, 0], loess_fit[:, 1], linewidth=2)
ax.axvline(PHQ_CUTOFF, linewidth=1, linestyle="--")
ax.set_xlabel("PHQ-8 total score")
ax.set_ylabel("Participant-level mean SADI")
ax.set_title("(a) SADI and depression severity", fontweight="bold")
ax.grid(axis="y", alpha=0.25)

# Panel B
ax = axes[1]
ax.violinplot(
    [non_depressed, depressed],
    positions=[1, 2],
    showmeans=False,
    showmedians=True,
    showextrema=True,
)
ax.set_xticks([1, 2])
ax.set_xticklabels(["PHQ-8 < 10", "PHQ-8 ≥ 10"])
ax.set_ylabel("Participant-level mean SADI")
ax.set_title("(b) SADI by PHQ-8 group", fontweight="bold")
ax.grid(axis="y", alpha=0.25)

fig.suptitle("Clinical Relevance of Semantic–Affective Discordance", fontsize=13, fontweight="bold")
fig.tight_layout()

figure_png = FIGURE_DIR / "Figure_5_PHQ8_SADI.png"
figure_svg = FIGURE_DIR / "Figure_5_PHQ8_SADI.svg"
fig.savefig(figure_png, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(figure_svg, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved PNG:", figure_png.resolve())
print("Saved SVG:", figure_svg.resolve())

## Interpretation guardrail

This analysis tests **clinical relevance/construct association**, not diagnostic validity. A significant correlation would support an association between naturally occurring participant-level SADI and self-reported symptom severity. Group differences likewise do not make SADI a diagnostic marker.